In [15]:
# pobieramy potrzebne biblioteki 

import os
import json
from openai import OpenAI
from dotenv import load_dotenv
import gradio as gr
import requests
from pypdf import PdfReader

load_dotenv(override=True)
myOpenai = OpenAI()


In [16]:

# pobieramy  klucze do api od aplikacji
# dla usera i token

# oraz url do połaczenia z http

pushoverUser = os.getenv("PUSHOVER_USER")
pushoverToken = os.getenv("PUSHOVER_TOKEN")
pushoverURL =  'https://api.pushover.net/1/messages.json'


In [17]:

# tworzymy funkcje push która będzie wysyłać wiadomość

def push(message):
    # tworzymy zmienna która zbiera dane do wiadomości wymagane przez api pushovera // dostepne w dokumentajci, dodajemy usera po pushoverUser, token po pushoverToken i message
    messages = {'user': pushoverUser, 'token': pushoverToken, 'message': message}
    # wysyłamy dane na serweh http pushover dzieki bibliotece request i metodzie post w argumentach wskazujemy url do serwera i data jako messages z naszą listą danych
    requests.post(pushoverURL, data=messages)
    

In [18]:

push('sama jestes cringe cebulaku')

In [19]:
# teraz funkcja która uruchamiana jest przez model jesli model ai moze odpowiedziec na pytanie usera, jako argumenty przekazujemy email, name i notes name i notes z przypisanymi wartosciami 
# jesli ktos ich nie uzupełni zeby nie wywaliło errora

def record_user_detail(email, name = 'unknown', notes = 'unknown'):
    # wywoyłjemy funkcje push i dorzucamy do niej dane z tej funkcji jako argument message poprostu przekazujemy ten string z mail name i notes do message i 
    push(f'{email},{name},{notes} ')
    # zwramy informacje w modelu json do modelu ai
    return {'recorded' : 'ok'}


In [20]:
# funkcja kóra sie uruchomi jesli model nie jest wstanie odpowiedzieć na pytanie usera

def record_unknown_question(question):

    # wywołujemy push i przeakzujemy doa rgumenty message w funkcju push nowy paramtetr
    push(f' {question} unknown')
    # zwracamy indofmacje w json do modelu ai
    return{'recorded': 'ok'}

In [21]:
# opis w json dla 1 tools w formacie json zgodny z api openAI 


record_user_details_json = {
    "name": "record_user_details",
    "description": "Use this tool to record that a user is interested in being in touch and provided an email address",
    "parameters": {
        "type": "object",
        "properties": {
            "email": {
                "type": "string",
                "description": "The email address of this user"
            },
            "name": {
                "type": "string",
                "description": "The user's name, if they provided it"
            }
            ,
            "notes": {
                "type": "string",
                "description": "Any additional information about the conversation that's worth recording to give context"
            }
        },
        "required": ["email"],
        "additionalProperties": False
    }
}

In [22]:
record_unknown_question_json = {
    "name": "record_unknown_question",
    "description": "Always use this tool to record any question that couldn't be answered as you didn't know the answer",
    "parameters": {
        "type": "object",
        "properties": {
            "question": {
                "type": "string",
                "description": "The question that couldn't be answered"
            },
        },
        "required": ["question"],
        "additionalProperties": False
    }
}

In [23]:


# tworzymy liste tools dla modelu ai  zgodnie z api openAI  type = funkcja, funkcja: nazwa instrukcji json dla danej funkcji

tools = [{'type': 'function','function': record_user_details_json},
        { 'type' : 'function', 'function': record_unknown_question_json}
        

]



